In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import boto3
import os
from tqdm import tqdm
import s3fs

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Helper Classes

In [ ]:
class TimeSeriesEDA:
    """Exploratory Data Analysis for time series."""
    
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.fs = s3fs.S3FileSystem()
        self.df_data = None
    
    def import_data(self, str_filename='employee_attrition_data.csv'):
        """Load data from S3."""
        str_uri = f's3://{self.str_bucket}/00_data_collection/{str_filename}'
        self.df_data = pd.read_csv(str_uri)
        self.df_data['date'] = pd.to_datetime(self.df_data['date'])
        self.df_data = self.df_data.sort_values('date').reset_index(drop=True)
        print(f'Loaded {len(self.df_data)} records from {str_uri}')
        return self.df_data
    
    def get_df_info(self):
        """Basic statistics and info."""
        print('\n=== Dataset Info ===')
        print(f'Shape: {self.df_data.shape}')
        print(f'Date range: {self.df_data["date"].min()} to {self.df_data["date"].max()}')
        print(f'Departments: {self.df_data["department"].nunique()}')
        print(f'Missing values: {self.df_data.isnull().sum().sum()}')
        
        print('\n=== Column Statistics ===')
        print(self.df_data.describe())
        
        print('\n=== Attrition by Department ===')
        df_by_dept = self.df_data.groupby('department').agg({
            'attrition_rate': ['mean', 'std'],
            'headcount': 'mean',
            'departures': 'sum'
        }).round(4)
        print(df_by_dept)
    
    def plot_attrition_over_time(self):
        """Line chart of company-wide attrition."""
        df_company = self.df_data.groupby('date')['attrition_rate'].mean().reset_index()
        
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.plot(df_company['date'], df_company['attrition_rate'], linewidth=2.5, 
                color='#d62728', marker='o', markersize=4, alpha=0.7)
        ax.fill_between(df_company['date'], df_company['attrition_rate'], alpha=0.3, color='#d62728')
        ax.set_xlabel('Date', fontsize=11)
        ax.set_ylabel('Attrition Rate', fontsize=11)
        ax.set_title('Company-Wide Monthly Attrition Rate Over Time', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/02_attrition_over_time.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved to {str_path}')
        plt.show()
    
    def plot_attrition_by_department(self):
        """Faceted line charts per department."""
        list_departments = sorted(self.df_data['department'].unique())
        int_n_depts = len(list_departments)
        int_cols = 3
        int_rows = (int_n_depts + int_cols - 1) // int_cols
        
        fig, axes = plt.subplots(int_rows, int_cols, figsize=(16, 3*int_rows))
        axes = axes.flatten()
        
        for int_idx, str_dept in enumerate(list_departments):
            df_dept = self.df_data[self.df_data['department'] == str_dept].sort_values('date')
            axes[int_idx].plot(df_dept['date'], df_dept['attrition_rate'], linewidth=2, marker='o', markersize=3)
            axes[int_idx].set_title(str_dept, fontsize=12, fontweight='bold')
            axes[int_idx].set_ylabel('Attrition Rate')
            axes[int_idx].grid(True, alpha=0.3)
            axes[int_idx].tick_params(axis='x', rotation=45)
        
        # Hide extra subplots
        for int_idx in range(int_n_depts, len(axes)):
            axes[int_idx].set_visible(False)
        
        plt.tight_layout()
        str_path = f'{self.str_dirname_output}/03_attrition_by_department.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved to {str_path}')
        plt.show()
    
    def plot_seasonal_decomposition(self):
        """Decompose time series into trend, seasonal, residual."""
        df_company = self.df_data.groupby('date')['attrition_rate'].mean().reset_index()
        df_company = df_company.set_index('date')
        
        # Seasonal decomposition with period=12 (monthly seasonality)
        result = seasonal_decompose(df_company['attrition_rate'], model='additive', period=12)
        
        fig, axes = plt.subplots(4, 1, figsize=(14, 10))
        
        axes[0].plot(df_company.index, df_company['attrition_rate'], linewidth=1.5)
        axes[0].set_ylabel('Original')
        axes[0].set_title('Time Series Decomposition (Additive Model)', fontsize=13, fontweight='bold')
        axes[0].grid(True, alpha=0.3)
        
        axes[1].plot(result.trend.index, result.trend.values, linewidth=1.5, color='#ff7f0e')
        axes[1].set_ylabel('Trend')
        axes[1].grid(True, alpha=0.3)
        
        axes[2].plot(result.seasonal.index, result.seasonal.values, linewidth=1.5, color='#2ca02c')
        axes[2].set_ylabel('Seasonal')
        axes[2].grid(True, alpha=0.3)
        
        axes[3].plot(result.resid.index, result.resid.values, linewidth=1.5, color='#d62728')
        axes[3].set_ylabel('Residual')
        axes[3].set_xlabel('Date')
        axes[3].grid(True, alpha=0.3)
        
        plt.tight_layout()
        str_path = f'{self.str_dirname_output}/04_seasonal_decomposition.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved to {str_path}')
        plt.show()
    
    def plot_headcount_trend(self):
        """Overall headcount growth over time."""
        df_hc = self.df_data.groupby('date')['headcount'].sum().reset_index()
        
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.plot(df_hc['date'], df_hc['headcount'], linewidth=2.5, color='#1f77b4', marker='o', markersize=4)
        ax.fill_between(df_hc['date'], df_hc['headcount'], alpha=0.2, color='#1f77b4')
        ax.set_xlabel('Date', fontsize=11)
        ax.set_ylabel('Total Headcount', fontsize=11)
        ax.set_title('Company-Wide Total Headcount Over Time', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/05_headcount_trend.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved to {str_path}')
        plt.show()
    
    def plot_monthly_boxplots(self):
        """Box plot of attrition rate by month."""
        list_data = [self.df_data[self.df_data['month'] == int_m]['attrition_rate'].values 
                     for int_m in range(1, 13)]
        
        fig, ax = plt.subplots(figsize=(14, 6))
        bp = ax.boxplot(list_data, labels=range(1, 13), patch_artist=True)
        
        for patch in bp['boxes']:
            patch.set_facecolor('#1f77b4')
            patch.set_alpha(0.6)
        
        ax.set_xlabel('Month', fontsize=11)
        ax.set_ylabel('Attrition Rate', fontsize=11)
        ax.set_title('Distribution of Attrition Rate by Month (Seasonal Pattern)', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/06_monthly_boxplots.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved to {str_path}')
        plt.show()
    
    def plot_autocorrelation(self):
        """ACF and PACF plots."""
        df_company = self.df_data.groupby('date')['attrition_rate'].mean().reset_index()
        series = df_company['attrition_rate'].values
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        plot_acf(series, lags=40, ax=axes[0])
        axes[0].set_title('Autocorrelation Function (ACF)', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Lag')
        
        plot_pacf(series, lags=40, ax=axes[1])
        axes[1].set_title('Partial Autocorrelation Function (PACF)', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Lag')
        
        plt.tight_layout()
        str_path = f'{self.str_dirname_output}/07_autocorrelation.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved to {str_path}')
        plt.show()
    
    def plot_rolling_statistics(self):
        """Rolling mean and std to visualize stationarity."""
        df_company = self.df_data.groupby('date')['attrition_rate'].mean().reset_index()
        series = df_company['attrition_rate']
        
        flt_rolling_mean = series.rolling(window=12).mean()
        flt_rolling_std = series.rolling(window=12).std()
        
        fig, ax = plt.subplots(figsize=(14, 6))
        
        ax.plot(series.index, series.values, label='Original', linewidth=1.5, alpha=0.7)
        ax.plot(flt_rolling_mean.index, flt_rolling_mean.values, label='12-Month Rolling Mean', 
                linewidth=2, color='#ff7f0e')
        ax.fill_between(flt_rolling_std.index, 
                         flt_rolling_mean - flt_rolling_std, 
                         flt_rolling_mean + flt_rolling_std, 
                         alpha=0.2, color='#ff7f0e', label='Std Dev Band')
        
        ax.set_xlabel('Time Index', fontsize=11)
        ax.set_ylabel('Attrition Rate', fontsize=11)
        ax.set_title('Rolling Mean and Standard Deviation (Stationarity Assessment)', fontsize=13, fontweight='bold')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/08_rolling_statistics.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved to {str_path}')
        plt.show()

## Constants

In [ ]:
str_bucket = 'time-series-forecasting-demo-repo'
str_task = 'employee_attrition_forecasting'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

print(f'Output directory: {str_dirname_output}')

## Load Data

In [ ]:
eda = TimeSeriesEDA(str_bucket, str_dirname_output)
df_data = eda.import_data()
eda.get_df_info()

## Attrition Over Time

In [ ]:
eda.plot_attrition_over_time()

## Attrition by Department

In [ ]:
eda.plot_attrition_by_department()

## Seasonal Decomposition

In [ ]:
eda.plot_seasonal_decomposition()

## Headcount Trend

## Monthly Seasonal Pattern

In [ ]:
eda.plot_monthly_boxplots()

## Autocorrelation Analysis

In [ ]:
eda.plot_autocorrelation()

## Rolling Statistics

In [ ]:
eda.plot_rolling_statistics()

## EDA Complete

In [ ]:
print('\n=== EDA SUMMARY ===')
print('Key Findings:')
print('1. Strong seasonal pattern with peaks in Jan/Feb/Sep')
print('2. Slight upward trend in overall attrition')
print('3. Department-specific rates: Sales highest, Engineering lowest')
print('4. Rolling mean shows relative stationarity with seasonal variation')
print('5. ACF/PACF suggest need for differencing or seasonal AR parameters')
print('\nAll visualizations saved to ./output/')